# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04** </center>
---
<br>
    <center>Saul Razo Magallanes - 739974</center>
    <center>Ingeniería en Sistemas Computacionales</center>
    <center><strong>Profesor:</strong> Pablo Camarillo Ramírez</center>
    <center><strong>Fecha:</strong> 02/03/2026</center>

In [15]:
from pcamarillor.spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Example - Extract info from JSON", "spark://spark-master:7077")

### Dirección de Datasets

In [16]:
!ls /opt/spark/work-dir/data/car_service

agencies  brands  cars	customers  rentals


In [17]:
from pyspark.sql.functions import get_json_object, col

### Creación de Dataframes

#### Agencies Dataframe

In [18]:
columns_types = [("agency_id", "int"),
                 ("agency_info", "string"),
                 ]
aggencies_schema = SparkUtils.generate_schema(columns_types)
agencies_df = su._spark.read \
                .option("header", "true") \
                .schema(aggencies_schema) \
                .csv("/opt/spark/work-dir/data/car_service/agencies/")

agencies_df = agencies_df \
    .withColumn("agency_name", get_json_object(col("agency_info"), "$.agency_name")) \
    .withColumn("city",        get_json_object(col("agency_info"), "$.city"))

#### Brands Dataframe

In [19]:
columns_types = [("brand_id", "int"),
                 ("brand_info", "string"),
                 ]
brands_schema = SparkUtils.generate_schema(columns_types)
brands_df = su._spark.read \
                .option("header", "true") \
                .schema(brands_schema) \
                .csv("/opt/spark/work-dir/data/car_service/brands/")

brands_df = brands_df \
    .withColumn("brand_name", get_json_object(col("brand_info"), "$.brand_name")) \
    .withColumn("country",    get_json_object(col("brand_info"), "$.country"))

#### Cars Dataframe

In [20]:
columns_types = [("car_id", "int"),
                 ("car_info", "string"),
                 ]
cars_schema = SparkUtils.generate_schema(columns_types)
cars_df = su._spark.read \
                .option("header", "true") \
                .schema(cars_schema) \
                .csv("/opt/spark/work-dir/data/car_service/cars/")

cars_df = cars_df \
    .withColumn("car_name",      get_json_object(col("car_info"), "$.car_name")) \
    .withColumn("brand_id",      get_json_object(col("car_info"), "$.brand_id").cast("int")) \
    .withColumn("price_per_day", get_json_object(col("car_info"), "$.price_per_day").cast("double"))

#### Customers Dataframe

In [21]:
columns_types = [("customer_id", "int"),
                 ("customer_info", "string"),
                 ]
customers_schema = SparkUtils.generate_schema(columns_types)
customers_df = su._spark.read \
                .option("header", "true") \
                .schema(customers_schema) \
                .csv("/opt/spark/work-dir/data/car_service/customers/")

customers_df = customers_df \
    .withColumn("customer_name", get_json_object(col("customer_info"), "$.customer_name")) \
    .withColumn("city",          get_json_object(col("customer_info"), "$.city")) \
    .withColumn("age",           get_json_object(col("customer_info"), "$.age").cast("int"))

#### Rentals Dataframe

In [22]:
columns_types = [("rental_id", "int"),
                 ("rental_info", "string"),
                 ]
rentals_schema = SparkUtils.generate_schema(columns_types)
rentals_df = su._spark.read \
                .option("header", "true") \
                .schema(rentals_schema) \
                .csv("/opt/spark/work-dir/data/car_service/rentals/")

rentals_df = rentals_df \
    .withColumn("car_id",      get_json_object(col("rental_info"), "$.car_id").cast("int")) \
    .withColumn("customer_id", get_json_object(col("rental_info"), "$.customer_id").cast("int")) \
    .withColumn("agency_id",   get_json_object(col("rental_info"), "$.agency_id").cast("int"))

### Final Dataframe

In [27]:
final_df = rentals_df \
    .join(cars_df,      rentals_df.car_id      == cars_df.car_id,      "left") \
    .join(agencies_df,  rentals_df.agency_id   == agencies_df.agency_id,  "left") \
    .join(customers_df, rentals_df.customer_id == customers_df.customer_id, "left") \
    .select(
        rentals_df.rental_id,
        cars_df.car_name,
        agencies_df.agency_name,
        customers_df.customer_name
    ) \
    .fillna({
        "car_name":      "Unknown",
        "agency_name":   "Unknown",
        "customer_name": "Unknown"
    }) \
    .orderBy("rental_id")
final_df.show(truncate=False)
print(f"Total de registros: {final_df.count()}")

+---------+------------------------------------+-------------+----------------+
|rental_id|car_name                            |agency_name  |customer_name   |
+---------+------------------------------------+-------------+----------------+
|0        |Grimes-Green Model 8                |SF Cars      |Laura Perry     |
|1        |Wallace-Carlson Model 9             |SF Cars      |Melanie Patrick |
|2        |Wagner LLC Model 1                  |NYC Rentals  |Theresa Estrada |
|3        |Bolton, Burns and Turner Model 10   |LA Car Rental|Amanda Garcia   |
|4        |Alvarez-Davis Model 3               |SF Cars      |Corey Cook      |
|5        |Grimes-Green Model 8                |SF Cars      |Kyle Ramos      |
|6        |Grimes-Green Model 8                |NYC Rentals  |Julie Chen      |
|7        |Clayton-Cook Model 10               |LA Car Rental|Sarah Peterson  |
|8        |Garcia, Hamilton and Carr Model 5   |SF Cars      |Jay Walsh       |
|9        |Chang-Fisher Model 7         

## HDFS| Formato Parquet

In [28]:
final_df.write \
    .mode("overwrite") \
    .partitionBy("agency_name") \
    .parquet("hdfs://namenode:9000/output/agencies_car_service")

26/03/03 03:01:58 WARN FileSystem: Failed to initialize filesystem hdfs://namenode:9000/output/agencies_car_service: java.lang.IllegalArgumentException: java.net.UnknownHostException: namenode


IllegalArgumentException: java.net.UnknownHostException: namenode

## Namenode UI 

In [13]:
su._spark.stop()